# Model 1 - Retrain from the unified `jobs` collection

The title->skills model trains from **one collection**: `jobs`, inside the `jobs`
database. Every posting the model has ever seen lives there, and the `source`
field is the axis that separates where each one came from:

| `source` | postings | what it is | how it gets there |
|---|---|---|---|
| `lang-uk` | 41,745 | Djinni corpus, 2020-01 -> 2023-09 | one-off `migrate_to_jobs.py` |
| `Linkedin` | grows nightly | LinkedIn scrape, 2026+ | `run_daily.sh` appends |

**No synthetic data.** `augmented-2026` - the 10,800 marked synthetic records that
used to bridge 2023H2->2026H1 - is retired: the nightly scrape now supplies the
recent end of the timeline with real postings. Every record that trains this model
is a real job posting, so there is no synthetic slice left to disclose.

**Append-only.** The nightly run upserts new postings into `jobs` and retrains.
Nothing rewrites the historical slice; section 1 verifies that directly.

**Before the first run** the historical corpus must be migrated in:

```
MONGO_URI=mongodb://.../jobs SOURCE_URI=mongodb://.../careerlens \
  DRY_RUN=0 python migrate_to_jobs.py
```

This notebook never extracts skills - it profiles what exists in `jobs` and
retrains from it.

## 0. Nightly training configuration

This notebook is the **source of truth** for how Model 1 is trained. The values
below are the ones section 5 reasons about, and `pipeline/run_daily.sh` reads
them straight out of this cell (via `pipeline/nightly_config.py`, which parses
the notebook with `ast.literal_eval` and never executes it), so the nightly run
and this notebook cannot drift apart.

Change a value here and the next nightly run picks it up. Nothing else needs
editing.

| key | why it is not the shipped default |
|---|---|
| `RECENCY_HALF_LIFE_DAYS` | default 14. Against a corpus reaching back to 2020 it decays the historical slice to nothing - see section 5. |
| `TREND_WINDOW_DAYS` | default 7. Almost no (role, skill) pair has an observation that recent, so every trend label collapses to `stable`. |
| `SOURCE_EXCLUDE` | keeps the retired `augmented-2026` synthetic bridge out of training. |
| `SOURCE_WEIGHTS` | one collection, one weight: age is handled once, by the half-life. |
| `TRAIN_USE_UNIFIED` | `0` on purpose - `SOURCE_EXCLUDE` is honoured only by the legacy accumulator, so the unified path would let the synthetic records back in under a real source's name. |

An operator can still override any single key for one run by exporting it in the
environment; `nightly_config.py` emits `${KEY:-<notebook value>}`.

In [ ]:
# The nightly pipeline reads this dict out of the notebook - keep it a plain
# literal (pipeline/nightly_config.py parses it with ast.literal_eval, so no
# expressions, f-strings or references to other variables).
NIGHTLY_TRAINING_CONFIG = {
    'MONGO_COLLECTION':        'jobs',
    'SOURCE_WEIGHTS':          'jobs:1.0',
    'SOURCE_EXCLUDE':          'augmented-2026',
    'RECENCY_HALF_LIFE_DAYS':  '365',
    'TREND_WINDOW_DAYS':       '365',
    'TRAIN_USE_UNIFIED':       '0',
    'UNIFIED_SKILLS_COLLECTION': 'role_skill_observations',
}

In [ ]:
import os, re, sys, subprocess, warnings
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymongo import MongoClient

warnings.filterwarnings('ignore')

# In production MONGO_URI already points at the `jobs` database.
os.environ.setdefault('MONGO_URI', 'mongodb://localhost:27017/jobs')
if os.path.basename(os.getcwd()) == 'final':     # notebook lives in ds/final;
    os.chdir(os.path.join('..', 'model'))        # code + data stay in ds/model
sys.path.insert(0, os.path.abspath('.'))

# NOTE: never `import train` here - train.py runs a full training at import time.
# The notebook shells out to it in section 5 instead.
from taxonomy import CANONICAL_TITLES, VARIANT_TO_CANONICAL

MONGO_URI = os.environ['MONGO_URI']
db = MongoClient(MONGO_URI, serverSelectionTimeoutMS=8000).get_default_database()

JOBS = 'jobs'
HISTORICAL, CURRENT = 'lang-uk', 'linkedin'
COLOR = {HISTORICAL: '#0f766e', CURRENT: '#c2610c'}   # teal = historical, orange = current
DEFAULT_COLOR = '#94a3b8'

def src_of(doc):
    # The scraper writes 'Linkedin' with a capital L; the migration writes
    # 'lang-uk'. Compare lower-cased everywhere - never on the raw value.
    return (doc.get('source') or 'unknown').strip().lower()

# Sources this retrain must NOT learn from. `augmented-2026` is the curated
# synthetic bridge that stood in for 2023H2->2026H1 before the scrape existed;
# now that source=linkedin covers that period with real postings it must not
# contribute. It comes from NIGHTLY_TRAINING_CONFIG above and is handed to train.py below, so the
# corpus this notebook profiles is exactly the corpus that trains.
EXCLUDE_SOURCES = [s.strip().lower() for s in
                   NIGHTLY_TRAINING_CONFIG['SOURCE_EXCLUDE'].split(',') if s.strip()]

# Mongo filter reused by every read below. Matching is case-insensitive because
# the scraper writes 'Linkedin' while the migration writes 'lang-uk'.
Q = ({'$expr': {'$not': [{'$in': [
    {'$toLower': {'$ifNull': ['$source', '']}}, EXCLUDE_SOURCES]}]}}
     if EXCLUDE_SOURCES else {})

if JOBS not in set(db.list_collection_names()):
    raise SystemExit(
        "collection 'jobs' not found in " + MONGO_URI.split('@')[-1] +
        ". train.py reads ONE database - point MONGO_URI at the database that "
        "holds the unified collection (production: .../jobs).")

by_source = {r['_id']: r['n'] for r in db[JOBS].aggregate([
    {'$match': Q},
    {'$group': {'_id': {'$toLower': {'$ifNull': ['$source', 'unknown']}},
                'n': {'$sum': 1}}}])}
total = db[JOBS].estimated_document_count()
print('database :', MONGO_URI.split('@')[-1])
print('jobs     :', format(total, ','), 'documents')
for s, n in sorted(by_source.items(), key=lambda kv: -kv[1]):
    print('   source=' + s.ljust(18), format(n, '>9,'))

if HISTORICAL not in by_source:
    print()
    print('!! no source=lang-uk documents - the historical corpus has not been')
    print('   migrated in. Run migrate_to_jobs.py before retraining, or this')
    print('   model will be trained on the scrape alone.')
if CURRENT not in by_source:
    print()
    print('!! no source=linkedin documents - the nightly scrape has not landed here.')

excluded_n = sum(n for s, n in by_source.items() if s in EXCLUDE_SOURCES)
if EXCLUDE_SOURCES:
    print()
    print('EXCLUDED from this retrain:', EXCLUDE_SOURCES,
          '->', format(excluded_n, ','), 'documents')
    print('training corpus:', format(total - excluded_n, ','), 'documents')
    still_there = [s for s in EXCLUDE_SOURCES if s in by_source]
    if not still_there and excluded_n == 0:
        print('   (none present - nothing to exclude)')


## 1. Is the collection actually unified, and is it append-only?

Three questions this section answers before anything else:

1. **Did the migration land intact?** The historical slice must be exactly 41,745
   documents and must never move again.
2. **Which roles does each source carry**, and which roles does the scrape *add*
   that the historical corpus never had? The old model covered 12 of 59 canonical
   roles - every added role is one the product could not rank skills for.
3. **Are there `og_title` values that are not canonical?** `train.py` drops those
   postings silently, so they are collected data that never reaches the model.

In [ ]:
EXPECTED_HISTORICAL = 41_745   # size of lang-uk-job-skills at migration time

hist = by_source.get(HISTORICAL, 0)
print('historical slice:', format(hist, ','), 'of', format(EXPECTED_HISTORICAL, ','), 'expected',
      '-> OK' if hist == EXPECTED_HISTORICAL else '-> MISMATCH')
if hist and hist != EXPECTED_HISTORICAL:
    print('   the historical slice changed size. It is append-only by contract:')
    print('   either the migration is incomplete, or something rewrote it.')

# _id prefixes prove which pipeline wrote each document, and that the two never collide.
prefixes = Counter()
for d in db[JOBS].find(Q, {'_id': 1}):
    _id = str(d['_id'])
    prefixes[_id.split(':', 1)[0] + ':' if ':' in _id else '(no prefix)'] += 1
print()
print('_id prefixes:', dict(prefixes))

In [ ]:
rows = []
for r in db[JOBS].aggregate([
        {'$match': Q},
        {'$group': {'_id': {'role': '$og_title',
                            'src': {'$toLower': {'$ifNull': ['$source', 'unknown']}}},
                    'n': {'$sum': 1}}}]):
    rows.append({'role': r['_id']['role'], 'source': r['_id']['src'], 'postings': r['n']})
prof = pd.DataFrame(rows)

pivot = prof.pivot_table(index='role', columns='source', values='postings',
                         aggfunc='sum', fill_value=0).astype(int)
for c in (HISTORICAL, CURRENT):
    if c not in pivot.columns:
        pivot[c] = 0
pivot['total'] = pivot.sum(axis=1)
pivot['canonical'] = [r in CANONICAL_TITLES for r in pivot.index]
pivot = pivot.sort_values('total', ascending=False)

added = pivot[(pivot[HISTORICAL] == 0) & (pivot[CURRENT] > 0) & pivot['canonical']]
non_canon = pivot[~pivot['canonical']]

print('roles with historical data:', int((pivot[HISTORICAL] > 0).sum()))
print('roles with current data   :', int((pivot[CURRENT] > 0).sum()))
print('canonical roles the scrape ADDS:', len(added), '->', list(added.index))
print('union coverage:', int((pivot['total'] > 0).sum()), 'of', len(CANONICAL_TITLES),
      'canonical roles')
if len(non_canon):
    print()
    print('!!', len(non_canon), 'og_title values are NOT canonical.', end=' ')
    print('train.py DROPS these postings')
    print('   unless their `title` matches a known variant. Fix the scrape keywords or')
    print('   add a mapping, otherwise this data is collected and never used:')
    print(non_canon[[HISTORICAL, CURRENT, 'total']].to_string())
pivot[[HISTORICAL, CURRENT, 'total']]

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(4, 0.32 * len(pivot))))
order = pivot.sort_values('total').index
y = np.arange(len(order))
ax.barh(y, pivot.loc[order, HISTORICAL], color=COLOR[HISTORICAL],
        label='source=lang-uk (2020-2023)')
ax.barh(y, pivot.loc[order, CURRENT], left=pivot.loc[order, HISTORICAL],
        color=COLOR[CURRENT], label='source=Linkedin (2026+)')
ax.set_yticks(y, order, fontsize=9)
ax.set_xlabel('postings in `jobs`')
ax.set_title('The unified collection, by role and source')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

### 1.1 Date-type audit - the silent one

The migration writes `datePosted` as a **BSON date**; the LinkedIn scraper writes
an **ISO string** (`"2026-06-29T06:53:17.000Z"`). Both live in the same collection.

`train.py` copes - `_parse_dt` accepts either - but naive notebook code
(`isinstance(x, datetime)`) silently drops every string-dated document, which
would make the entire current slice vanish from the analysis below without
raising anything. Every date in this notebook goes through `as_dt()`.

To eliminate the mixed types entirely, run the migrator with
`NORMALIZE_TARGET_DATES=1`.

In [ ]:
def as_dt(v):
    # BSON date, ISO-8601 string ('...Z' included), or None -> tz-aware datetime
    if isinstance(v, datetime):
        return v if v.tzinfo else v.replace(tzinfo=timezone.utc)
    if isinstance(v, str) and v:
        try:
            return datetime.fromisoformat(v.replace('Z', '+00:00'))
        except ValueError:
            return None
    return None

audit = {}
for d in db[JOBS].find(Q, {'datePosted': 1, 'source': 1}):
    s = src_of(d)
    a = audit.setdefault(s, {'types': Counter(), 'unparsable': 0, 'missing': 0, 'lo': None, 'hi': None})
    raw = d.get('datePosted')
    a['types'][type(raw).__name__] += 1
    dt = as_dt(raw)
    if dt is None:
        # Absent (None / missing) and corrupt (present but unparsable) are very
        # different: train.py tolerates an absent date - _bucket_month simply
        # skips that posting's stability bucket - but a value it cannot parse
        # means the field is the wrong shape and every date-derived number for
        # that slice is suspect. Only the second is a stop condition.
        if raw is None:
            a['missing'] += 1
        else:
            a['unparsable'] += 1
        continue
    a['lo'] = dt if a['lo'] is None or dt < a['lo'] else a['lo']
    a['hi'] = dt if a['hi'] is None or dt > a['hi'] else a['hi']

audit_df = pd.DataFrame([
    {'source': s, 'datePosted_types': dict(a['types']),
     'missing': a['missing'], 'unparsable': a['unparsable'],
     'earliest': a['lo'].date() if a['lo'] else None,
     'latest': a['hi'].date() if a['hi'] else None}
    for s, a in sorted(audit.items())])
print(audit_df.to_string(index=False))

n_missing = int(audit_df['missing'].sum())
if n_missing:
    print()
    print(format(n_missing, ',') + ' postings carry NO datePosted. They still train,')
    print('but they contribute nothing to the stability slope or the trend window,')
    print('and they are invisible on every time axis in this notebook.')

assert audit_df['unparsable'].sum() == 0, (
    'datePosted values that exist but cannot be parsed: '
    + str(int(audit_df['unparsable'].sum()))
    + '. Fix the field shape before training - every date-derived number '
      'for that slice would be wrong.')


## 2. Extraction quality per source

The same SkillNer pipeline runs over two very different text distributions. This
section measures, per source: skills per posting, the full-match / n-gram split,
and how much n-gram output is surface noise.

Noise matters more than it looks. `build_skill_records` returns a document's
stored `skill_records` **as-is** when present, so `train.py`'s `is_valid_skill`
guard never runs on pre-extracted documents - whatever SkillNer wrote at ingest
time is what trains the model.

In [ ]:
def doc_skills(doc, min_ngram_score=0.9):
    # distinct skill strings for one document: full matches + high-confidence n-grams
    seen = set()
    sk = doc.get('skills') or {}
    for m in sk.get('full_matches', []):
        v = (m.get('doc_node_value') or '').lower().strip()
        if len(v) >= 3:
            seen.add(v)
    for m in sk.get('ngram_matches', []):
        v = (m.get('doc_node_value') or '').lower().strip()
        if len(v) >= 3 and float(m.get('score', 0)) >= min_ngram_score:
            seen.add(v)
    return seen

records, per_doc = [], []
for doc in db[JOBS].find(Q, {'og_title': 1, 'skills': 1, 'datePosted': 1, 'source': 1}):
    s = src_of(doc)
    dt = as_dt(doc.get('datePosted'))
    month = dt.strftime('%Y-%m') if dt else None
    sk = doc_skills(doc)
    raw = doc.get('skills') or {}
    per_doc.append({'source': s, 'role': doc.get('og_title'), 'month': month,
                    'n_skills': len(sk),
                    'n_full': len(raw.get('full_matches', [])),
                    'n_ngram': len(raw.get('ngram_matches', []))})
    for skill in sk:
        records.append((s, doc.get('og_title'), skill, month))

eda = pd.DataFrame(records, columns=['source', 'role', 'skill', 'month'])
docs = pd.DataFrame(per_doc)
docs['raw_matches'] = docs['n_full'] + docs['n_ngram']

summary = docs.groupby('source').agg(
    documents=('n_skills', 'size'),
    zero_skill_docs=('n_skills', lambda s: int((s == 0).sum())),
    median_skills=('n_skills', 'median'),
    mean_skills=('n_skills', 'mean'),
    median_raw_matches=('raw_matches', 'median'),
    below_5_raw=('raw_matches', lambda s: int((s < 5).sum())),
).round(2)
summary['extraction_coverage_pct'] = (
    100 * (1 - summary['zero_skill_docs'] / summary['documents'])).round(2)
print(format(len(eda), ',') + ' (doc, skill) observations |',
      format(eda['skill'].nunique(), ','), 'distinct skills')
summary

In [ ]:
# n-gram noise probe. Two-token n-grams made of ordinary verbs/nouns
# ('environments manage', 'projects implement') are SkillNer surface artefacts,
# not skills. Reported, never silently filtered - the fix belongs in
# extract_skills.py at ingest time, not in a training notebook.
STOPISH = {'manage', 'managing', 'implement', 'implementing', 'experience', 'work',
           'working', 'strong', 'environments', 'environment', 'projects', 'project',
           'secure', 'data', 'servers', 'office', 'team', 'teams', 'business',
           'knowledge', 'skills', 'requirements', 'solutions', 'support'}

def looks_like_noise(skill):
    toks = skill.split()
    return len(toks) == 2 and all(t in STOPISH for t in toks)

counters = {}
for doc in db[JOBS].find(Q, {'skills.ngram_matches': 1, 'source': 1}):
    c = counters.setdefault(src_of(doc), Counter())
    for m in (doc.get('skills') or {}).get('ngram_matches', []):
        v = (m.get('doc_node_value') or '').lower().strip()
        if len(v) >= 3:
            c[v] += 1

ngram_rows = []
for s, c in sorted(counters.items()):
    tot = sum(c.values())
    noisy = sum(n for k, n in c.items() if looks_like_noise(k))
    worst = [k for k, _ in c.most_common(300) if looks_like_noise(k)][:6]
    ngram_rows.append({'source': s, 'ngram_observations': tot, 'suspected_noise': noisy,
                       'noise_pct': round(100 * noisy / max(tot, 1), 2),
                       'examples': ', '.join(worst)})
pd.DataFrame(ngram_rows)

## 3. Skills x role x time

With the scrape supplying 2026 and the historical corpus supplying 2020-2023, the
timeline is real end to end for the first time. The gap between the two slices is
left visible and is not filled in - that gap is the honest picture of what the
corpus does and does not cover.

In [ ]:
top_by_role = (eda.groupby(['role', 'skill']).size().rename('docs').reset_index()
               .sort_values(['role', 'docs'], ascending=[True, False])
               .groupby('role').head(12))
roles = sorted(r for r in top_by_role['role'].dropna().unique())
nrows = max(1, (len(roles) + 2) // 3)
fig, axes = plt.subplots(nrows, 3, figsize=(18, 4 * nrows), squeeze=False)
for ax, role in zip(axes.flat, roles):
    g = top_by_role[top_by_role['role'] == role].set_index('skill')['docs'].sort_values()
    dom = eda[eda['role'] == role].groupby(['skill', 'source']).size().unstack(fill_value=0)
    for c in (HISTORICAL, CURRENT):
        if c not in dom.columns:
            dom[c] = 0
    cols = [COLOR[CURRENT] if dom.loc[s, CURRENT] > dom.loc[s, HISTORICAL]
            else COLOR[HISTORICAL] for s in g.index]
    g.plot(kind='barh', ax=ax, color=cols)
    ax.set_title(role, fontsize=11)
    ax.set_ylabel('')
for ax in axes.flat[len(roles):]:
    ax.axis('off')
plt.suptitle('Top-12 skills per role  (teal = mostly historical, orange = mostly current)',
             y=1.001, fontsize=14)
plt.tight_layout(); plt.show()

### 3.1 Reading a time axis across two corpora

The collection spans two corpora that differ in **role mix**, **volume** and
**era**, with a multi-year hole between them. Three rules follow, and section 3.2
measures each one rather than assuming it.

**1. Never plot raw counts.** A count of skill observations per month measures
how much was scraped that month, not what the market wanted. The historical slice
carries hundreds of postings a month; the scrape carries a fraction of that. On a
count axis every skill appears to collapse at the boundary - an artefact of
collection volume, not of demand.

**2. Normalise inside the role, not across the corpus.** Global prevalence (share
of *all* postings that month) silently encodes the role mix. The historical corpus
is developer roles; the scrape leans wherever its search keywords point. A skill
can appear to crash purely because the mix changed. The fix is a **role-balanced**
rate: compute the share inside each role, then average those shares across a fixed
basket of roles present in **both** slices. Role mix then cannot move the number.

**3. A regression across the gap is not a trend.** With no data between the two
eras, a fitted slope is determined by the width of the hole, and its p-value is
meaningless. Where the eras are compared, the honest instrument is a **two-period
comparison** with both sample sizes shown - not a line, not a slope.

Monthly curves stay valid **inside** the historical slice, which has the volume to
support them. Section 3.3 keeps them there and labels them as such.

In [ ]:
# 3.2a  The denominator, first. Every rate below divides by these numbers, so
# they are shown before any rate is drawn.
docs_m = docs.dropna(subset=['month']).copy()
by_month_src = docs_m.groupby(['month', 'source']).size().unstack(fill_value=0)
for c in (HISTORICAL, CURRENT):
    if c not in by_month_src.columns:
        by_month_src[c] = 0
span = pd.period_range(min(by_month_src.index), max(by_month_src.index), freq='M').astype(str)
by_month_src = by_month_src.reindex(span, fill_value=0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=True)

ax1.bar(span, by_month_src[HISTORICAL], color=COLOR[HISTORICAL], label='source=lang-uk')
ax1.bar(span, by_month_src[CURRENT], bottom=by_month_src[HISTORICAL],
        color=COLOR[CURRENT], label='source=linkedin')
ax1.set_ylabel('postings')
ax1.set_title('A. Collection volume per month - the denominator of every rate below')
ax1.legend(frameon=False)
ax1.spines[['top', 'right']].set_visible(False)

# 3.2b  Skill density: distinct skills per posting. This is the one 'quantity of
# skills over time' curve that is legitimate month by month - but it measures
# posting richness and extractor behaviour, NOT market demand.
dens = docs_m.groupby(['month', 'source'])['n_skills'].median().unstack()
for c in [c for c in (HISTORICAL, CURRENT) if c in dens.columns]:
    d = dens[c].reindex(span)
    ax2.plot(span, d.values, color=COLOR[c], linewidth=2, marker='.', label='source=' + c)
ax2.set_ylabel('median skills / posting')
ax2.set_title('B. Skill density per posting (extraction richness, not demand)')
ax2.legend(frameon=False)
ax2.spines[['top', 'right']].set_visible(False)
ax2.set_xticks([m for i, m in enumerate(span) if i % 3 == 0])
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

empty = [m for m in span if by_month_src.loc[m].sum() == 0]
print('timeline:', span[0], '->', span[-1], '(' + str(len(span)), 'months)')
print('months with NO postings in either slice:', len(empty),
      ('(' + empty[0] + ' .. ' + empty[-1] + ')') if empty else '')
print()
def median_active(col):
    # median over months that HAVE postings. An empty slice yields NaN, which
    # `or 0` does not catch (NaN is truthy) - a single-source corpus is a normal
    # state (before the migration, or before the scrape lands), not an error.
    active = by_month_src[by_month_src[col] > 0][col]
    v = active.median()
    return 0 if pd.isna(v) else int(v)

print('median postings/month  historical:', median_active(HISTORICAL),
      '| current:', median_active(CURRENT))
for col, label in ((HISTORICAL, 'historical'), (CURRENT, 'current')):
    if by_month_src[col].sum() == 0:
        print('   note: the', label, 'slice is EMPTY - every comparison below')
        print('   degrades to single-source and section 3.2c will report no basket.')

In [ ]:
# 3.2c  Role-balanced two-period comparison - the only instrument here that
# supports a claim about the market.
MIN_ROLE_DOCS = 25          # per (role, period); below this the share is noise

roles_hist = set(docs_m[docs_m['source'] == HISTORICAL]['role'].dropna())
roles_curr = set(docs_m[docs_m['source'] == CURRENT]['role'].dropna())
BASKET = sorted(roles_hist & roles_curr)
print('role basket (present in BOTH slices):', BASKET or 'EMPTY')

eda_m = eda.dropna(subset=['month']).copy()
period = {HISTORICAL: docs_m['source'] == HISTORICAL, CURRENT: docs_m['source'] == CURRENT}

den = {p: docs_m[m & docs_m['role'].isin(BASKET)].groupby('role').size()
       for p, m in period.items()}
usable = {p: [r for r in BASKET if den[p].get(r, 0) >= MIN_ROLE_DOCS] for p in den}
COMPARABLE = sorted(set(usable[HISTORICAL]) & set(usable[CURRENT]))
print('roles clearing the', MIN_ROLE_DOCS, 'posting floor in BOTH periods:',
      COMPARABLE or 'NONE')

if not COMPARABLE:
    print()
    print('>> No role has enough postings in both periods, so no role-balanced')
    print('>> comparison can be made yet. This is a VOLUME limit, not an absence')
    print('>> of signal: keep scraping, or widen the period, and re-run.')
    for p in (HISTORICAL, CURRENT):
        print('   ', p, 'postings per basket role:', dict(den[p].reindex(BASKET, fill_value=0)))
else:
    rows = []
    for p, mask in period.items():
        d_idx = docs_m[mask & docs_m['role'].isin(COMPARABLE)]
        e_idx = eda_m[eda_m['role'].isin(COMPARABLE) &
                      eda_m['month'].isin(set(d_idx['month']))]
        dn = d_idx.groupby('role').size()
        for skill, grp in e_idx.groupby('skill'):
            shares = []
            cnt = grp.groupby('role').size()
            for r in COMPARABLE:
                if dn.get(r, 0) >= MIN_ROLE_DOCS:
                    shares.append(cnt.get(r, 0) / dn[r])
            if shares:
                rows.append({'skill': skill, 'period': p, 'prevalence': np.mean(shares)})
    cmp_df = (pd.DataFrame(rows).pivot(index='skill', columns='period', values='prevalence')
              .fillna(0.0))
    cmp_df['delta_pp'] = 100 * (cmp_df[CURRENT] - cmp_df[HISTORICAL])
    cmp_df = cmp_df[(cmp_df[[HISTORICAL, CURRENT]].max(axis=1) >= 0.03)]

    top = pd.concat([cmp_df.nlargest(12, 'delta_pp'), cmp_df.nsmallest(12, 'delta_pp')])
    fig, ax = plt.subplots(figsize=(10, max(5, 0.35 * len(top))))
    y = np.arange(len(top))
    ax.hlines(y, 100 * top[HISTORICAL], 100 * top[CURRENT], color='#cbd5e1', linewidth=2)
    ax.scatter(100 * top[HISTORICAL], y, color=COLOR[HISTORICAL], s=45, zorder=3,
               label='historical (lang-uk)')
    ax.scatter(100 * top[CURRENT], y, color=COLOR[CURRENT], s=45, zorder=3,
               label='current (scrape)')
    ax.set_yticks(y, top.index, fontsize=9)
    ax.set_xlabel('share of postings in the role basket (%)')
    ax.set_title('Role-balanced prevalence: historical vs current\\n'
                 'basket = ' + ', '.join(COMPARABLE))
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout(); plt.show()

    # A skill sitting at EXACTLY 0.0 on one side is almost always a corpus
    # difference (vocabulary, posting language, extractor behaviour), not a
    # market move. Flag it instead of letting it top the chart as a 'trend'.
    top = top.copy()
    top['caveat'] = np.where(
        top[[HISTORICAL, CURRENT]].min(axis=1) == 0,
        'ABSENT one side - corpus difference, not a trend', '')
    n_absent = int((top['caveat'] != '').sum())
    if n_absent:
        print()
        print('!!', n_absent, 'of', len(top), 'rows are absent on one side entirely.')
        print('   Read those as corpus differences, not as market movement.')

    print()
    print('n per role, historical:', dict(den[HISTORICAL].reindex(COMPARABLE, fill_value=0)))
    print('n per role, current   :', dict(den[CURRENT].reindex(COMPARABLE, fill_value=0)))
    display(top.rename(columns={HISTORICAL: 'historical', CURRENT: 'current'}).round(4))

In [ ]:
# 3.3  Monthly prevalence curves, restricted to the HISTORICAL slice, where the
# volume supports them. Nothing is drawn across the gap: the current slice is
# deliberately absent here - section 3.2c is the like-for-like comparison.
hist_docs = docs_m[docs_m['source'] == HISTORICAL]
hist_months = hist_docs.groupby('month').size()

if hist_months.empty:
    # A corpus with no historical slice is a normal state (the migration has not
    # run yet, or training is deliberately scrape-only), not an error. But the
    # time-derived features will be weak, so name the consequences rather than
    # drawing an empty chart.
    print('No source=' + HISTORICAL + ' postings: skipping the monthly series.')
    print()
    print('Expect this from the retrain below:')
    print('  * stability_score falls back to its neutral default wherever a')
    print('    (role, skill) spans fewer than 2 distinct week buckets;')
    print('  * every posting sits inside TREND_WINDOW_DAYS, so recent prevalence')
    print('    equals all-time prevalence and trend labels collapse to "stable";')
    print('  * roles that leaned on the historical slice can fall under the')
    print('    promotion-gate thresholds - section 6 names them.')
else:
    hist_eda = eda_m[eda_m['month'].isin(set(hist_docs['month']))]
    WATCH = ['python', 'react', 'docker', 'kubernetes', 'typescript', 'php',
             'jquery', 'machine learning', 'aws', 'sql']
    plt.figure(figsize=(15, 5))
    drawn = 0
    for sk in WATCH:
        m = (hist_eda[hist_eda['skill'] == sk].groupby('month').size() / hist_months).dropna()
        m = m[m.index >= '2020-01'].sort_index()
        if len(m) > 3:
            plt.plot(m.index, m.values, linewidth=2, marker='.', markersize=3, label=sk)
            drawn += 1
    if drawn:
        plt.legend(ncol=5, fontsize=9)
    plt.title('Monthly skill prevalence - HISTORICAL slice only (the slice with the volume)')
    ticks = sorted(hist_months.index)
    plt.xticks([m for i, m in enumerate(ticks) if i % 3 == 0], rotation=45)
    plt.tight_layout(); plt.show()
    print('historical slice:', ticks[0], '->', ticks[-1], '|',
          format(int(hist_months.sum()), ','), 'postings |', drawn, 'skills drawn')
    print('Current-slice values are NOT plotted here - see 3.2c for the')
    print('like-for-like comparison, which controls for role mix.')

## 4. What the training filters will keep

`train.py` drops a posting when (a) `resolve_canonical` cannot map it, or (b) it
carries fewer than **5** raw SkillNer matches - a constant hard-coded in
`accumulate_from_collection`, not an env var.

Simulating both here means the retrain below holds no surprises, and the
per-source split shows whether scraped postings survive at the same rate as the
historical ones.

In [ ]:
MIN_RAW_MATCHES = 5   # train.py, accumulate_from_collection

def canonical_of(doc):
    # Mirrors train.py's resolve_canonical exactly. Grouping by raw og_title is
    # WRONG here: a posting tagged 'Python Developer' trains as Software
    # Engineer, so a per-og_title table does not predict what train.py prints.
    og = doc.get('og_title') or doc.get('og_tite')
    if og in CANONICAL_TITLES:
        return og
    t = (doc.get('title') or '').lower().strip()
    if t and t in VARIANT_TO_CANONICAL:
        return VARIANT_TO_CANONICAL[t]
    return None


def resolves(doc):
    return canonical_of(doc) is not None

acc = {}
for doc in db[JOBS].find(Q, {'og_title': 1, 'title': 1, 'skills': 1, 'source': 1}):
    a = acc.setdefault(src_of(doc), {'documents': 0, 'dropped_no_canonical_title': 0,
                                     'dropped_below_5_matches': 0, 'trains': 0})
    a['documents'] += 1
    if not resolves(doc):
        a['dropped_no_canonical_title'] += 1
        continue
    sk = doc.get('skills') or {}
    if len(sk.get('full_matches', [])) + len(sk.get('ngram_matches', [])) < MIN_RAW_MATCHES:
        a['dropped_below_5_matches'] += 1
        continue
    a['trains'] += 1

fn = pd.DataFrame([dict(source=s, **a) for s, a in sorted(acc.items())])
fn['retention_pct'] = (100 * fn['trains'] / fn['documents']).round(2)
print(fn.to_string(index=False))
print()
print('TOTAL records that will train:', format(int(fn['trains'].sum()), ','))
for _, row in fn.iterrows():
    if row['retention_pct'] < 90:
        print('!!', row['source'], 'loses', round(100 - row['retention_pct'], 1),
              'percent of its documents - inspect before promoting')

## 4.1 Pre-train diagnostics

`train.py` is a long job that builds every accumulator before it touches the
feature matrix, so a data problem surfaces at the *end* of the run. Everything
here reproduces the conditions that make it fail - cheaply, and before it starts.

Three failures this catches:

* **Mixed datetime provenance.** `build_skill_records` returns a document's
  stored `skill_records` unchanged when present, and pymongo hands those back
  **naive**. A document *without* stored records has them recomputed through
  `_parse_dt`, which returns **tz-aware**. When one role receives both,
  `compute_stability_score` raises
  `TypeError: can't compare offset-naive and offset-aware datetimes`.
* **A source that never reaches the accumulators.** If `SOURCE_WEIGHTS` points
  at the wrong collection, training silently proceeds on whatever it did find.
  The per-source record table below is what `train.py`'s own
  "Records loaded per role" output must match.
* **Postings collected but never used** - `og_title` values outside the
  taxonomy, and postings under the 5-match floor.

In [ ]:
# ── 1. Is the tz-normalisation fix present in this checkout? ─────────────────
try:
    from skill_schema import _as_utc                      # noqa: F401
    print('skill_schema : tz normalisation present')
    FIX_PRESENT = True
except ImportError:
    FIX_PRESENT = False
    print('!! skill_schema has NO _as_utc - this checkout predates the fix.')
    print("   compute_stability_score will raise TypeError: can't compare")
    print('   offset-naive and offset-aware datetimes as soon as one role')
    print('   receives observations from both provenances (see below).')

# ── 2. Datetime provenance per source ───────────────────────────────────────
# A document WITH stored skill_records contributes naive dates (BSON -> pymongo).
# A document WITHOUT them has dates recomputed via _parse_dt -> tz-aware.
# Only documents that SURVIVE the training filters can contribute observations,
# so the provenance mix must be measured after them - counting every document
# over-reports badly (in the historical corpus 967 documents lack skill_records,
# but 966 of them are dropped at the 5-match floor and never reach the sort).
prov = []
role_prov = {}
skipped_pre_filter = 0
for d in db[JOBS].find(Q, {'og_title': 1, 'title': 1, 'source': 1,
                            'skill_records': 1, 'skills': 1}):
    sk = d.get('skills') or {}
    if not resolves(d) or (len(sk.get('full_matches', [])) +
                           len(sk.get('ngram_matches', []))) < MIN_RAW_MATCHES:
        skipped_pre_filter += 1
        continue
    recs = d.get('skill_records')
    stored = isinstance(recs, list) and len(recs) > 0
    prov.append({'source': src_of(d), 'stored_records': stored})
    r = d.get('og_title')
    if r:
        role_prov.setdefault(r, {'naive': 0, 'aware': 0})
        role_prov[r]['naive' if stored else 'aware'] += 1
print('(' + format(skipped_pre_filter, ',') +
      ' documents dropped by the training filters are excluded below)')

prov_df = (pd.DataFrame(prov).groupby('source')['stored_records']
           .agg(documents='size',
                with_stored_records='sum',
                without=lambda x: int((~x).sum())))
prov_df['provenance'] = np.where(
    prov_df['without'] == 0, 'all naive (stored)',
    np.where(prov_df['with_stored_records'] == 0, 'all aware (recomputed)', 'MIXED'))
print()
print(prov_df.to_string())

mixed_roles = {r: v for r, v in role_prov.items() if v['naive'] and v['aware']}
print()
if mixed_roles:
    print('roles receiving BOTH provenances:', len(mixed_roles))
    for r, v in sorted(mixed_roles.items(), key=lambda kv: -sum(kv[1].values()))[:10]:
        print('   ' + r.ljust(34), 'stored=' + str(v['naive']).rjust(6),
              'recomputed=' + str(v['aware']).rjust(6))
    if not FIX_PRESENT:
        raise SystemExit(
            'These roles WILL crash compute_stability_score on this checkout. '
            'Pull the skill_schema.py fix (_as_utc) before retraining.')
    print('   -> handled: _as_utc normalises both to UTC before sorting.')
else:
    print('no role mixes provenances - the tz crash cannot trigger on this corpus.')

# ── 3. What each source will actually contribute, per role ──────────────────
# This table is what train.py's own "Records loaded per role" must match. A row
# that is non-zero here but zero there means SOURCE_WEIGHTS missed a collection.
contrib = []
remapped = Counter()
for d in db[JOBS].find(Q, {'og_title': 1, 'title': 1, 'skills': 1, 'source': 1}):
    canon = canonical_of(d)
    if canon is None:
        continue
    sk = d.get('skills') or {}
    if len(sk.get('full_matches', [])) + len(sk.get('ngram_matches', [])) < MIN_RAW_MATCHES:
        continue
    contrib.append({'role': canon, 'source': src_of(d)})
    og = d.get('og_title')
    if og and og != canon:
        remapped[(og, canon)] += 1

if contrib:
    ct = (pd.DataFrame(contrib).groupby(['role', 'source']).size()
          .unstack(fill_value=0))
    ct['total'] = ct.sum(axis=1)
    ct = ct.sort_values('total', ascending=False)
    print()
    print('records each source will contribute (post-filter):')
    print(ct.to_string())
    print()
    print('These are CANONICAL totals - the same grouping train.py prints, so the')
    print('two must match. A source column that is all zeros means SOURCE_WEIGHTS')
    print('does not cover that slice.')

    if remapped:
        rm = (pd.DataFrame([{'og_title': o, 'trains_as': c, 'postings': n}
                            for (o, c), n in remapped.most_common()])
              .sort_values('postings', ascending=False))
        print()
        print(format(int(rm['postings'].sum()), ','), 'postings carry an og_title that is')
        print('not canonical and are folded into another role via their `title`.')
        print('Worth checking that each fold is intended - the scrape keyword is')
        print('the cheapest place to fix a wrong one:')
        display(rm.head(25))
else:
    raise SystemExit('nothing survives the filters - training would produce an empty model.')

## 5. Retrain

`train.py` stays the single source of truth; the notebook only points it at the
collection. One collection means one entry in `SOURCE_WEIGHTS`:

```
SOURCE_WEIGHTS=jobs:1.0
```

Both slices therefore carry the same source weight, and age is handled once - by
the recency half-life below - instead of being discounted twice.

> If the two slices ever need *different* weights, `SOURCE_WEIGHTS` cannot express
> it any more: it maps collection names, and there is now only one collection.
> That would need a small additive change in `train.py` - a `SOURCE_FIELD_WEIGHTS`
> env resolved against each document's `source` field. Not wired today, because
> both slices are at 1.0.

**`RECENCY_HALF_LIFE_DAYS=365` / `TREND_WINDOW_DAYS=365`** are not optional. The
shipped defaults (14 / 7) were built for a continuously-scraped corpus anchored to
"now". Against a corpus reaching back to 2020 they annihilate all history: a 2023
posting would weigh `0.5^76` (about 1e-23) against about 0.08 for a 2026 posting,
making the model effectively scrape-only - and the promotion gate would pass it,
because the gate counts records, not weights. At 365 days a 2023 posting weighs
about 0.13 against about 0.91: recent demand leads, history still counts.

WARNING: `TREND_WINDOW_DAYS=365` also defines "recent" for the
rising/stable/falling label. Since the historical slice ends in 2023, only scraped
postings fall inside that window, so `trend` reads as "current demand vs the
blended all-time baseline". For roles the scrape does not cover, recent equals
all-time and every skill lands on `stable`. Section 6 reports that per role rather
than hiding it.

`model.joblib` is backed up first; `train.py` writes a versioned snapshot and
promotes only if the promotion gate passes.

In [ ]:
import shutil
if os.path.exists('model.joblib'):
    existing = [f for f in os.listdir('.') if f.startswith('model.joblib.bak-')]
    if not existing:
        bak = 'model.joblib.bak-' + datetime.now().strftime('%Y%m%d_%H%M%S')
        shutil.copy2('model.joblib', bak)
        print('backup:', bak)
    else:
        print('backup already exists:', existing)
    shutil.copy2('model.joblib', 'model.joblib.baseline')   # for the section-6 diff

env = dict(os.environ)
# Exactly what run_daily.sh will export tonight - same dict, same values.
env.update(NIGHTLY_TRAINING_CONFIG)
env.update({'MONGO_URI': MONGO_URI, 'PYTHONIOENCODING': 'utf-8'})
r = subprocess.run([sys.executable, 'train.py'], capture_output=True, text=True, env=env)
print(r.stdout[-4000:])

# train.py exits NON-ZERO when the promotion gate blocks the new model - that is
# a designed outcome (it is how run_daily.sh knows to skip the DS restart), not a
# failure. Treat it as such: the snapshot was still written and is still worth
# inspecting. Only a genuine crash should stop the notebook.
snap = re.search(r'Saved versioned snapshot:\s*(.+\.joblib)', r.stdout)
blocked = re.search(r'NOT promoted \(([^)]*)\)', r.stdout)

if r.returncode == 0:
    PROMOTED, GATE_REASON = True, 'promoted'
    MODEL_UNDER_TEST = 'model.joblib'
elif blocked and snap:
    PROMOTED, GATE_REASON = False, blocked.group(1)
    MODEL_UNDER_TEST = snap.group(1).strip()
    print()
    print('=' * 72)
    print('GATE BLOCKED PROMOTION:', GATE_REASON)
    print('model.joblib is UNCHANGED. Section 6 below verifies the new snapshot')
    print('instead, so what you inspect is what was actually just trained:')
    print('   ', MODEL_UNDER_TEST)
    print('=' * 72)
else:
    raise SystemExit('train.py failed (not a gate block): '
                     + (r.stderr[-2000:] or r.stdout[-2000:]))

## 6. Verification

Loads the newly written `model.joblib` and checks: per-role record counts, the
trend-label distribution, the top-5 skills each role will actually serve (through
`select_display_skills`, i.e. including the commonness filter), and a diff against
the pre-retrain baseline.

In [ ]:
import joblib
art = joblib.load(MODEL_UNDER_TEST)
print('verifying:', MODEL_UNDER_TEST,
      '(PROMOTED)' if PROMOTED else '(snapshot - gate blocked)')
fm = art['feature_matrix']
print('trained_at:', art['trained_at'])

# What train.py STORED vs what the product actually SERVES.
#
# train.py labels trends with fixed ratio thresholds (rise >= 1.25, fall <= 0.80).
# server.py never uses those labels: recalibrate_trend_labels() rewrites every one
# of them in memory at load time, from the percentiles of the ratio distribution
# that actually exists. So reading f['trend'] out of the joblib measures a field
# no user ever sees - on this corpus it is 100% 'stable' by construction. Both
# views are reported below; the SERVED one is the one that matters.
stored_counts = Counter(f.get('trend', '?') for role in fm.values() for f in role.values())


def served_trend_labels(matrix):
    """Mirror of server.py's recalibrate_trend_labels. Kept in step by hand: if
    the server's percentiles or floors change, change them here too."""
    ratios = []
    for skills in matrix.values():
        for f in skills.values():
            p, rp = f.get('prevalence') or 0, f.get('recent_prevalence')
            if p > 0 and rp is not None and rp > 0:
                ratios.append(rp / p)
    if len(ratios) < 100:
        return {}, {'relabeled': False, 'reason': 'only %d usable ratios' % len(ratios)}
    ratios.sort()
    rise_cut = max(ratios[int(len(ratios) * 0.80)], 1.05)
    fall_cut = min(ratios[int(len(ratios) * 0.20)], 0.95)
    labels, counts = {}, Counter()
    for role, skills in matrix.items():
        for sk, f in skills.items():
            p, rp = f.get('prevalence') or 0, f.get('recent_prevalence')
            if p > 0 and rp is not None and rp > 0:
                ratio = rp / p
                lab = ('rising' if ratio >= rise_cut
                       else 'falling' if ratio <= fall_cut else 'stable')
            else:
                lab = 'stable'
            labels[(role, sk)] = lab
            counts[lab] += 1
    return labels, {'relabeled': True, 'rise_cut': round(rise_cut, 4),
                    'fall_cut': round(fall_cut, 4), 'counts': dict(counts),
                    'usable_ratios': len(ratios)}


SERVED_TREND, RECAL = served_trend_labels(fm)
populated = {r: len(v) for r, v in fm.items() if v}
print('roles with skills:', len(populated), 'of', len(CANONICAL_TITLES),
      '| (role, skill) rows:', format(sum(populated.values()), ','))
print()
print('trend AS STORED  (train.py, fixed 1.25/0.80):', dict(stored_counts))
print('trend AS SERVED  (server.py recalibration)  :', RECAL)

# THE number that decides everything: how many (role, skill) rows even HAVE a
# recent observation. Rows without one are hard-coded 'stable' by both the stored
# labels and the recalibration, so if coverage is low no threshold change can
# help - the fix is to get postings INTO the trend window.
total_rows = sum(len(v) for v in fm.values())
usable = RECAL.get('usable_ratios', 0)
print()
print('rows with a usable recent_prevalence: %s of %s (%.1f%%)'
      % (format(usable, ','), format(total_rows, ','), 100 * usable / max(total_rows, 1)))
if usable < 0.1 * total_rows:
    print('   >> under 10%. Every other row serves as "stable" regardless of the')
    print('      cuts. Before touching thresholds, check what is keeping postings')
    print('      out of the window: section 1.1 lists datePosted types per source,')
    print('      and a posting with a NULL datePosted can never count as recent.')

# Why the labels look the way they do: the spread of recent/overall prevalence.
# If that distribution is flat, NO thresholding scheme can produce a trend and
# the fix is upstream - TREND_WINDOW_DAYS, or a corpus spanning more than one era.
ratios = np.array([(f['recent_prevalence'] / f['prevalence'])
                   for role in fm.values() for f in role.values()
                   if (f.get('prevalence') or 0) > 0
                   and f.get('recent_prevalence') is not None
                   and f['recent_prevalence'] > 0])
print()
if len(ratios):
    p1, p20, p50, p80, p99 = np.percentile(ratios, [1, 20, 50, 80, 99])
    print('recent/overall prevalence ratio spread over', format(len(ratios), ','), 'skills:')
    print('   p1=%.3f  p20=%.3f  median=%.3f  p80=%.3f  p99=%.3f'
          % (p1, p20, p50, p80, p99))
    if p80 < 1.05 and p20 > 0.95:
        print('   >> the whole distribution sits inside the server floors (1.05 /')
        print('      0.95), so every skill still serves as "stable" even AFTER')
        print('      recalibration. That is a corpus/window problem, not a')
        print('      labelling one: TREND_WINDOW_DAYS must make "recent" a real')
        print('      subset of the corpus, otherwise recent == overall by')
        print('      construction and the ratio is 1.0 everywhere.')
else:
    print('no usable ratios: recent_prevalence is missing or zero everywhere,')
    print('so no rule can produce a trend. Check TREND_WINDOW_DAYS against the')
    print('corpus date range printed in section 1.1.')

# Per role, using the SERVED labels. A role is signal-less only when none of its
# skills has a usable recent_prevalence - not merely when everything is stable.
rows = []
for r, v in fm.items():
    if not v:
        continue
    c = Counter(SERVED_TREND.get((r, sk), 'stable') for sk in v)
    usable = sum(1 for f in v.values()
                 if (f.get('prevalence') or 0) > 0
                 and f.get('recent_prevalence') is not None
                 and f['recent_prevalence'] > 0)
    rows.append({'role': r, 'skills': len(v), 'rising': c['rising'],
                 'stable': c['stable'], 'falling': c['falling'],
                 'skills_with_recent_data': usable})
tr = pd.DataFrame(rows).set_index('role')
tr['trend_signal'] = np.where(
    tr['skills_with_recent_data'] == 0, 'NONE (no recent observations)',
    np.where(tr['rising'] + tr['falling'] == 0, 'flat (all inside the cuts)', 'present'))
none_roles = list(tr.index[tr['trend_signal'].str.startswith('NONE')])
print()
print('roles with NO recent observations:', len(none_roles), none_roles[:8],
      '...' if len(none_roles) > 8 else '')
print('roles whose served labels are all "stable":',
      int((tr['trend_signal'] == 'flat (all inside the cuts)').sum()))
tr.sort_values('skills', ascending=False)


In [ ]:
# Which titles crossed the gate's threshold? The rule is zero-tolerance: if the
# number of titles holding >= NON_LOW_THRESHOLD records falls by even one against
# the last PROMOTED run, promotion is blocked - however much everything else
# improved. train.py records every run in `model_runs`, so the diff is exact rather
# than inferred; this names the titles instead of leaving the message
# ('non_low titles dropped 51->50') as a bare number.
from promotion_gate import NON_LOW_THRESHOLD

runs = list(db['model_runs'].find().sort([('_id', -1)]).limit(50))
this_run = next((r for r in runs if r.get('trained_at') == art['trained_at']), None)
last_promoted = next((r for r in runs
                      if r.get('promoted') and r.get('trained_at') != art['trained_at']), None)

if not this_run or not last_promoted:
    print('need both this run and a previous promoted run in model_runs to diff '
          'the gate; found this_run=' + str(bool(this_run)) +
          ' last_promoted=' + str(bool(last_promoted)))
else:
    new_counts = {t: int(n) for t, n in this_run.get('record_counts', {}).items()}
    baseline = {t: int(n) for t, n in last_promoted.get('record_counts', {}).items()}
    n_old = sum(1 for t in CANONICAL_TITLES if baseline.get(t, 0) >= NON_LOW_THRESHOLD)
    n_new = sum(1 for t in CANONICAL_TITLES if new_counts.get(t, 0) >= NON_LOW_THRESHOLD)
    print('baseline run:', last_promoted.get('trained_at'),
          '| titles >=', NON_LOW_THRESHOLD, 'records:', n_old, '->', n_new)

    rows = []
    for t in CANONICAL_TITLES:
        o, n = baseline.get(t, 0), new_counts.get(t, 0)
        if o == n:
            continue
        was, now = o >= NON_LOW_THRESHOLD, n >= NON_LOW_THRESHOLD
        rows.append({'title': t, 'baseline': o, 'new': n, 'delta': n - o,
                     'crossed': 'FELL BELOW' if (was and not now)
                                else 'rose above' if (not was and now) else ''})
    diff = pd.DataFrame(rows)
    if len(diff):
        fell = diff[diff['crossed'] == 'FELL BELOW']
        if len(fell):
            print()
            print('THESE TITLES BLOCKED PROMOTION (dropped under',
                  NON_LOW_THRESHOLD, 'records):')
            print(fell.to_string(index=False))
            print()
            print('Options: collect more postings for them; or, if the drop is')
            print('expected, change NON_LOW_THRESHOLD / MIN_NON_LOW_TITLES')
            print('deliberately and record why. Do not disable the gate.')
        display(diff.sort_values('delta'))
    else:
        print('no per-title record counts changed.')

In [ ]:
from skill_schema import select_display_skills, compute_role_counts

def served_top5(feature_matrix):
    rc = compute_role_counts(feature_matrix)
    return {role: [s['skill'] for s in select_display_skills(
                feats, pool_size=10, display_count=5, role_counts=rc)]
            for role, feats in feature_matrix.items() if feats}

new_top = served_top5(fm)
base_top = {}
if os.path.exists('model.joblib.baseline'):
    base_top = served_top5(joblib.load('model.joblib.baseline')['feature_matrix'])

rows = []
for role in sorted(new_top):
    old = base_top.get(role)
    kept = str(len(set(new_top[role]) & set(old))) + '/5' if old else 'NEW ROLE'
    rows.append({'role': role, 'records': len(fm[role]),
                 'top5_served': ', '.join(new_top[role]), 'kept_from_baseline': kept})
res = pd.DataFrame(rows)
gained = [r['role'] for r in rows if r['kept_from_baseline'] == 'NEW ROLE']
churn = [r['role'] for r in rows if r['kept_from_baseline'] != 'NEW ROLE'
         and int(r['kept_from_baseline'][0]) < 3]
lost = sorted(set(base_top) - set(new_top))
print('roles gained vs baseline:', len(gained), gained)
print('roles LOST vs baseline :', len(lost), lost, '<-- must be empty' if lost else '')
print('roles whose top-5 changed by 3+ skills:', len(churn), churn)
pd.set_option('display.max_colwidth', 110)
res

In [ ]:
# Ubiquity cap sanity. SKILL_UBIQUITY_CAP drops skills appearing in more than N
# roles. The deployed value (11) was tuned for a 12-role model; as the role count
# grows the same absolute cap becomes far stricter in relative terms.
CURRENT_CAP = int(os.getenv('SKILL_UBIQUITY_CAP', '48'))
n_roles = len(populated)
rc = compute_role_counts(fm)
dist = Counter(rc.values())
filtered = sum(n for k, n in dist.items() if k > CURRENT_CAP)
print('roles with data:', n_roles, '| SKILL_UBIQUITY_CAP in this env:', CURRENT_CAP)
print('skills filtered at that cap:', format(filtered, ','), 'of', format(len(rc), ','))
if n_roles > 12:
    print('>> the deployed cap of 11 was tuned for 12 roles (ratio 0.92).')
    print('>> for', n_roles, 'roles the equivalent cap is', round(0.92 * n_roles),
          '- retune SKILL_UBIQUITY_CAP or common skills get over-filtered.')
pd.Series(dist).sort_index().rename('skills_in_N_roles').to_frame().T

In [ ]:
# Do the modern skills the synthetic corpus used to inject now arrive from REAL
# postings? This replaces the old EMERGING_SKILLS audit against market_2026_skills.
WAS_SYNTHETIC = ['llm', 'rag', 'playwright', 'kubernetes', 'terraform',
                 'pytorch', 'typescript', 'graphql']
found = {sk: sorted(r for r, feats in fm.items() if sk in feats) for sk in WAS_SYNTHETIC}
for sk, rs in found.items():
    print(' ', sk.ljust(12), str(len(rs)).rjust(2), 'roles',
          rs[:4] if rs else '  <-- ABSENT')
absent = [s for s, r in found.items() if not r]
print()
print(len(WAS_SYNTHETIC) - len(absent), 'of', len(WAS_SYNTHETIC),
      'formerly-synthetic skills now come from real postings',
      ('; still absent: ' + str(absent)) if absent else '')

## 7. Wrap-up

* `model.joblib` is trained from **one collection, `jobs`**, holding two real
  slices and nothing synthetic: `source='lang-uk'` (2020-2023 depth) and
  `source='Linkedin'` (current market, appended nightly).
* Recency half-life 365 days keeps the historical slice alive while letting
  current demand lead the ranking.
* Role coverage is no longer capped at the 12 the Djinni taxonomy could reach -
  the scrape adds every canonical role it covers (section 1).
* Serving verification (DS restart, `/title/skills`, the trend endpoint) runs
  outside the notebook.

**Open items, stated rather than discovered later:**
1. Roles whose only data is the scrape have no trend signal until a second year of
   scraping - section 6 lists them explicitly.
2. `SKILL_UBIQUITY_CAP` needs re-tuning as the role count grows (section 6).
3. n-gram noise (section 2) is written at ingest time; filtering it belongs in
   `extract_skills.py`, not here.
4. Any `og_title` the scrape produces that is not canonical is dropped silently by
   `train.py` - section 1 surfaces the count.
5. Mixed `datePosted` types (BSON date vs ISO string) are tolerated by `train.py`;
   `migrate_to_jobs.py NORMALIZE_TARGET_DATES=1` removes the mix.